# HIPE-2022 (fr) extraction + word-level OCR quality assessment

1. Load `HIPE-2022-v2.1-hipe2020-dev-fr.tsv` (French dev split) from the HIPE-2022-data repo.
2. Extract `document_id`, `token`, `NE-COARSE-LIT`, `MISC` into a flat DataFrame.
3. Run the impresso `OCR-quality-assessment-unigram` tool on each token to flag whether it is a known French word (a proxy for word-level OCR quality).

Note on the OCR-QA tool: 
- a bloom filter of known French word forms (built from Wikipedia + lexicons)
- Value: `True` = known word (likely correct OCR), `False` = not found (possible OCR error, or a rare/proper name), `None` = token normalized to nothing (pure punctuation).

In [17]:
# Run once per environment. On L3iCalcul: activate/create your conda env first (see run_job_cpu.sh),
# then run this cell, or pip install these ahead of time in your sbatch script.
%pip install -q pandas requests cython pybloomfiltermmap3 huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [43]:
import re
import unicodedata
from typing import Optional


import pandas as pd
import requests
from huggingface_hub import hf_hub_download
from pybloomfilter import BloomFilter

## 1. Load and parse the HIPE TSV

The file is CoNLL-style: a header row, `#`-prefixed comment/metadata lines (including `# hipe2022:document_id = ...` marking the start of each document), blank lines separating segments, and tab-separated token rows otherwise.

In [27]:
HIPE_URL = (
    "https://raw.githubusercontent.com/hipe-eval/HIPE-2022-data/main/"
    "data/v2.1/hipe2020/fr/HIPE-2022-v2.1-hipe2020-dev-fr.tsv"
)

DOC_ID_RE = re.compile(r"^#\s*hipe2022:document_id\s*=\s*(.+)$")


def load_hipe_tsv(url: str) -> pd.DataFrame:
    """Parse a HIPE-2022 TSV into a flat DataFrame of
    document_id / token / NE-COARSE-LIT / MISC rows."""
    text = requests.get(url, timeout=30).text
    lines = text.splitlines()

    header = lines[0].split("\t")
    idx_token = header.index("TOKEN")
    idx_ne_coarse_lit = header.index("NE-COARSE-LIT")
    idx_misc = header.index("MISC")

    rows = []
    current_doc_id = None
    for line in lines[1:]:
        if not line.strip():
            continue  # blank line = segment boundary within the same document
        if line.startswith("#"):
            m = DOC_ID_RE.match(line.strip())
            if m:
                current_doc_id = m.group(1).strip()
            continue
        fields = line.split("\t")
        rows.append(
            {
                "document_id": current_doc_id,
                "token": fields[idx_token],
                "NE-COARSE-LIT": fields[idx_ne_coarse_lit],
                "MISC": fields[idx_misc],
            }
        )
    return pd.DataFrame(rows)


df = load_hipe_tsv(HIPE_URL)
print(df.shape, "tokens across", df["document_id"].nunique(), "documents")

(37952, 4) tokens across 43 documents


In [58]:
# from transformers import pipeline

# MODEL_NAME = "impresso-project/ocr-quality-assessor-unigram-light"

# ocrqa_pipeline = pipeline("ocr-qa-assessment", model=MODEL_NAME, 
#                           trust_remote_code=True, 
#                           device='cpu')

# sentence = ""","""

# score = ocrqa_pipeline(sentence)
# print(score)


Loading model for en: models/ocrqa-wp_v1.0.5-en.bloom
Loading model for de: models/ocrqa-wp_v1.0.5-de.bloom
Loading model for fr: models/ocrqa-wp_v1.0.5-fr.bloom
Loading model for other: models/ocrqa-wp_v1.0.5-en.bloom
{'ocr_quality_score': 0.0}


## 2. Word-level OCR quality assessment (impresso bloom filter, French)

Uses `ocrqa-wp_v1.0.6-fr.bloom`, the French word-list bloom filter from
[impresso-project/OCR-quality-assessment-unigram](https://huggingface.co/impresso-project/OCR-quality-assessment-unigram).

In [30]:
# Normalization table as documented on the model card (NFKC + lowercase + digits->'0' + strip punctuation)
QUOTES_PUNCT = "\u201e\u2022<>!\"#%&'\u2019"
ASCII_PUNCT = "()*,./:;?"
BRACKETS_SPECIAL = "[]\\~_{}"
UNICODE_PUNCT = "\xa1\xab\xb7\xbb\xbf"
DASH_CARET = "\u2014^`"
SPECIAL_SYMBOLS = "\xa6\xa7\xa3="
HYPHEN = "-"
DIGITS = "0123456789"

NORMALIZATION_TABLE = str.maketrans(
    {
        char: " "
        for char in (
            QUOTES_PUNCT
            + ASCII_PUNCT
            + BRACKETS_SPECIAL
            + UNICODE_PUNCT
            + DASH_CARET
            + SPECIAL_SYMBOLS
            + HYPHEN
        )
    }
    | {char: "0" for char in DIGITS}
)


def normalize_text(s: str, unicode_normalize: Optional[str] = "NFKC") -> str:
    if unicode_normalize:
        s = unicodedata.normalize(unicode_normalize, s).lower()
    return s.translate(NORMALIZATION_TABLE)


def get_bloomfilter(model_id: str, filename: str) -> BloomFilter:
    return BloomFilter.open(hf_hub_download(repo_id=model_id, filename=filename))


bf_fr = get_bloomfilter(
    "impresso-project/OCR-quality-assessment-unigram", "ocrqa-wp_v1.0.6-fr.bloom"
)

In [48]:
def word_is_known(token: str, bloom_filter: BloomFilter) -> Optional[bool]:
    """True: token is a known French word (bloom filter hit).
    False: token normalized to a non-empty string but was not found (likely OCR error, or a rare/proper name).
    None: token normalized to nothing (pure punctuation/whitespace token) -- not applicable."""
    normalized = normalize_text(token).strip()
    if not normalized:
        return None
    sub_tokens = normalized.split()
    return all(t in bloom_filter for t in sub_tokens)


df["ocr_word_known"] = df["token"].apply(lambda t: word_is_known(t, bf_fr))
df.head(10)

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known
0,EXP-1888-08-03-a-i0030,FAITS,O,_,True
1,EXP-1888-08-03-a-i0030,DIVERS,O,EndOfLine,True
2,EXP-1888-08-03-a-i0030,La,O,_,True
3,EXP-1888-08-03-a-i0030,panique,O,_,True
4,EXP-1888-08-03-a-i0030,des,O,_,True
5,EXP-1888-08-03-a-i0030,éléphants,O,_,True
6,EXP-1888-08-03-a-i0030,au,O,_,True
7,EXP-1888-08-03-a-i0030,grand,O,EndOfLine,True
8,EXP-1888-08-03-a-i0030,cortège,O,_,True
9,EXP-1888-08-03-a-i0030,dc,O,_,True


In [32]:
df.describe(include="all")

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known
count,37952,37952,37952,37952,30683
unique,43,7896,11,14,2
top,GDL-1868-10-16-a-i0010,.,O,_,True
freq,2424,1869,34144,24423,30102


In [49]:
df[df.ocr_word_known == False].head(10)

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known
121,EXP-1888-08-03-a-i0030,wigsstrasse,B-loc,_,False
160,EXP-1888-08-03-a-i0030,phants,O,_,False
276,EXP-1888-08-03-a-i0030,pokets,O,NoSpaceAfter,False
338,EXP-1888-08-03-a-i0030,élép,O,_,False
342,EXP-1888-08-03-a-i0030,phants,O,_,False
435,EXP-1888-08-03-a-i0030,élép,O,_,False
483,EXP-1908-06-23-a-i0083,cotiférenefesont,O,_,False
496,EXP-1908-06-23-a-i0083,Hôfw,I-pers,_,False
507,EXP-1908-06-23-a-i0083,ôco,I-pers,NoSpaceAfter,False
559,EXP-1908-06-23-a-i0083,ociétés,O,_,False


In [60]:
df[df.ocr_word_known.isnull()].head(10)

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known
11,EXP-1888-08-03-a-i0030,.,O,EndOfSentence,None
12,EXP-1888-08-03-a-i0030,—,O,_,None
18,EXP-1888-08-03-a-i0030,",",O,_,None
22,EXP-1888-08-03-a-i0030,.,I-pers,EndOfLine,None
24,EXP-1888-08-03-a-i0030,",",I-pers,_,None
29,EXP-1888-08-03-a-i0030,',I-pers,NoSpaceAfter,None
33,EXP-1888-08-03-a-i0030,",",O,_,None
44,EXP-1888-08-03-a-i0030,",",O,_,None
51,EXP-1888-08-03-a-i0030,.,O,EndOfSentence,None
68,EXP-1888-08-03-a-i0030,.,O,EndOfLine|EndOfSentence,None


In [62]:
# impact of input is one token 
df.iloc[20:40]

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known,entity_type
20,EXP-1888-08-03-a-i0030,par,O,_,True,O
21,EXP-1888-08-03-a-i0030,M,B-pers,NoSpaceAfter,True,pers
22,EXP-1888-08-03-a-i0030,.,I-pers,EndOfLine,None,pers
23,EXP-1888-08-03-a-i0030,Hagenbeck,I-pers,NoSpaceAfter,True,pers
24,EXP-1888-08-03-a-i0030,",",I-pers,_,None,pers
25,EXP-1888-08-03-a-i0030,le,I-pers,_,True,pers
26,EXP-1888-08-03-a-i0030,célèbre,I-pers,_,True,pers
27,EXP-1888-08-03-a-i0030,marchand,I-pers,_,True,pers
28,EXP-1888-08-03-a-i0030,d,I-pers,NoSpaceAfter,True,pers
29,EXP-1888-08-03-a-i0030,',I-pers,NoSpaceAfter,None,pers


In [51]:
df["ocr_word_known"].value_counts(dropna=False)

ocr_word_known
True     30102
None      7269
False      581
Name: count, dtype: int64

In [61]:
# Entity type counts
print("Raw BIO tag counts:")
print(df["NE-COARSE-LIT"].value_counts())

def bio_entity_type(tag: str) -> str:
    """Strip the B-/I- prefix; 'O' stays 'O'."""
    return tag if tag == "O" else tag.split("-", 1)[1]

df["entity_type"] = df["NE-COARSE-LIT"].apply(bio_entity_type)

print("\nToken counts per entity type (B- and I- summed):")
print(df["entity_type"].value_counts())

Raw BIO tag counts:
NE-COARSE-LIT
O         34144
I-pers     1396
B-loc       774
B-pers      679
I-loc       268
I-org       246
B-org       159
I-time      112
B-time       68
I-prod       57
B-prod       49
Name: count, dtype: int64

Token counts per entity type (B- and I- summed):
entity_type
O       34144
pers     2075
loc      1042
org       405
time      180
prod      106
Name: count, dtype: int64


### OCR feature extraction (mention-span, context, document level)

Assume `None` (punctuation, not scoreable) as `True`, only actual `False` as unrecognized word.

- `entity_span_id`: groups consecutive `B-X`/`I-X` tokens into one mention (a new `B-` always starts a new span, even for the same type back-to-back); `NaN` for `O` tokens.
- `ocr_mean_mention`: mean OCR-known rate across the tokens of a mention's span.
- `ocr_ratio_context_5` / `ocr_ratio_context_10`: OCR-known rate in a centered window of ±5 / ±10 tokens, computed within each document (never crossing document boundaries).
- `doc_ocr_mean`: OCR-known rate over the whole document.

In [63]:
# Treat None (not scoreable / punctuation) as True for these aggregate features.
df["ocr_ok"] = df["ocr_word_known"].apply(lambda v: True if v is None else bool(v))

# Group consecutive B-X / I-X tokens into mention spans; O tokens get no span.
is_begin = df["NE-COARSE-LIT"].str.startswith("B-")
df["entity_span_id"] = is_begin.cumsum()
df.loc[df["NE-COARSE-LIT"] == "O", "entity_span_id"] = pd.NA

In [ ]:
# Mention-span features: mean OCR-known rate inside the span.
df["ocr_mean_mention"] = df.groupby("entity_span_id")["ocr_ok"].transform("mean")

In [67]:
# Context features: OCR-known rate in a centered +-5 / +-10 token window, per document.
df["ocr_ratio_context_5"] = df.groupby("document_id")["ocr_ok"].transform(
    lambda s: s.rolling(window=11, center=True, min_periods=1).mean()
)
df["ocr_ratio_context_10"] = df.groupby("document_id")["ocr_ok"].transform(
    lambda s: s.rolling(window=21, center=True, min_periods=1).mean()
)

# Document-level OCR-known rate.
df["doc_ocr_mean"] = df.groupby("document_id")["ocr_ok"].transform("mean")

In [ ]:
# Preview the new features on entity tokens only, where mention-span features are non-null.
df[df["entity_span_id"].notna()][
    ["document_id", "token", "NE-COARSE-LIT", "entity_span_id",
     "ocr_mean_mention", "ocr_low_ratio_mention",
     "ocr_ratio_context_5", "ocr_ratio_context_10", "doc_ocr_mean"]
].head(20)

In [68]:
df.iloc[20:30]

,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known,entity_type,ocr_ok,entity_span_id,ocr_mean_mention,ocr_ratio_context_5,ocr_ratio_context_10,doc_ocr_mean
20,EXP-1888-08-03-a-i0030,par,O,_,True,O,True,NaN,NaN,1.0,1.0,0.986667
21,EXP-1888-08-03-a-i0030,M,B-pers,NoSpaceAfter,True,pers,True,2.0,1.0,1.0,1.0,0.986667
22,EXP-1888-08-03-a-i0030,.,I-pers,EndOfLine,None,pers,True,2.0,1.0,1.0,1.0,0.986667
23,EXP-1888-08-03-a-i0030,Hagenbeck,I-pers,NoSpaceAfter,True,pers,True,2.0,1.0,1.0,1.0,0.986667
24,EXP-1888-08-03-a-i0030,",",I-pers,_,None,pers,True,2.0,1.0,1.0,1.0,0.986667
25,EXP-1888-08-03-a-i0030,le,I-pers,_,True,pers,True,2.0,1.0,1.0,1.0,0.986667
26,EXP-1888-08-03-a-i0030,célèbre,I-pers,_,True,pers,True,2.0,1.0,1.0,1.0,0.986667
27,EXP-1888-08-03-a-i0030,marchand,I-pers,_,True,pers,True,2.0,1.0,1.0,1.0,0.986667
28,EXP-1888-08-03-a-i0030,d,I-pers,NoSpaceAfter,True,pers,True,2.0,1.0,1.0,1.0,0.986667
29,EXP-1888-08-03-a-i0030,',I-pers,NoSpaceAfter,None,pers,True,2.0,1.0,1.0,1.0,0.986667


## 3. Analysis

In [52]:
applicable = df[df["ocr_word_known"].notna()]
print("Overall known-word rate:", applicable["ocr_word_known"].mean().round(3))

is_entity = applicable["NE-COARSE-LIT"] != "O"
print("Known-word rate on entity tokens:", applicable.loc[is_entity, "ocr_word_known"].mean().round(3))
print("Known-word rate on non-entity tokens:", applicable.loc[~is_entity, "ocr_word_known"].mean().round(3))

applicable.groupby("document_id")["ocr_word_known"].mean().sort_values().head(10)

Overall known-word rate: 0.981
Known-word rate on entity tokens: 0.942
Known-word rate on non-entity tokens: 0.986


document_id
GDL-1798-10-13-a-i0010    0.849206
GDL-1808-09-30-a-i0002    0.877615
GDL-1798-08-30-a-i0013    0.891129
GDL-1828-11-14-a-i0003    0.940945
GDL-1918-11-11-a-i0104    0.947368
GDL-1818-05-22-a-i0007    0.958904
GDL-1978-05-11-a-i0236    0.965986
GDL-1838-06-15-a-i0002    0.967667
GDL-1928-08-28-a-i0014    0.968397
GDL-1848-07-21-a-i0004    0.968831
Name: ocr_word_known, dtype: object

In [70]:
OUT_PATH = "hipe2020_dev_fr_ocrqa.csv"
df.to_csv(OUT_PATH, index=False)
print("Saved to", OUT_PATH)

Saved to hipe2020_dev_fr_ocrqa.csv


Problem: 
- This data seems like too clean compared to Le Temps 
- OCR word-level value: only True or False (Not the same as OCR confidence score). 2 options: only clear about sentence/document-level or continue using word-level (0/1 value) 